# Ejercicio 2: el 95% que no detecta nada

`pagos_taxi.csv`: 6000 pagos de la app de taxis, de septiembre y octubre de 2026. `es_fraude` vale 1 si el pago resultó ser fraudulento (tarjeta robada o contracargo). Mismo esqueleto del Ejercicio 1: `evaluar` corre una validación cruzada con el esquema que le pases y guarda una fila por modelo en `resultados`.

Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo dicen qué variable dejar. La última celda imprime una comprobación por paso; no la modifiques. Corre todas las celdas una vez antes de empezar (**Run All**).

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pagos = pd.read_csv("pagos_taxi.csv")
pagos["hora"] = pd.to_datetime(pagos["fecha_hora"]).dt.hour
pagos["monto_por_km"] = pagos["monto"] / pagos["distancia_km"]

NUMERICAS = ["monto", "distancia_km", "monto_por_km", "hora", "tarjeta_nueva", "intentos_fallidos", "viajes_previos"]
CATEGORICAS = ["metodo"]
X = pagos[NUMERICAS + CATEGORICAS]
y = pagos["es_fraude"]
print("Pagos:", len(pagos), " fraudes:", y.sum(), f" ({y.mean():.1%})")

Pagos: 6000  fraudes: 295  (4.9%)


## La función que evalúa cada modelo

In [2]:
resultados = {}


def preparacion():
    return ColumnTransformer([
        ("numericas", StandardScaler(), NUMERICAS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAS),
    ])


def evaluar(nombre, modelo, esquema):
    """Validacion cruzada del modelo con el esquema dado. Guarda en
    resultados el promedio de cada metrica entre los pliegues."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    puntajes = cross_validate(pipeline, X, y, cv=esquema, scoring=["accuracy", "recall", "precision"])
    resultados[nombre] = {
        "modelo": nombre,
        "accuracy": round(puntajes["test_accuracy"].mean(), 4),
        "recall": round(puntajes["test_recall"].mean(), 4),
        "precision": round(puntajes["test_precision"].mean(), 4),
    }
    print(f"{nombre:20s} accuracy {resultados[nombre]['accuracy']:.4f}   recall {resultados[nombre]['recall']:.4f}   precision {resultados[nombre]['precision']:.4f}")

## Paso 1: un esquema que respete la proporción de fraudes

In [3]:
esquema = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for numero, (_, indices_validacion) in enumerate(esquema.split(X, y)):
    print(f"Pliegue {numero + 1}: {y.iloc[indices_validacion].sum()} fraudes de {len(indices_validacion)}")


Pliegue 1: 59 fraudes de 1200
Pliegue 2: 59 fraudes de 1200
Pliegue 3: 59 fraudes de 1200
Pliegue 4: 59 fraudes de 1200
Pliegue 5: 59 fraudes de 1200


## Paso 2: la línea base

In [4]:
evaluar("linea base", DummyClassifier(strategy="most_frequent"), esquema)

linea base           accuracy 0.9508   recall 0.0000   precision 0.0000


/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/vscode/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/vscode/.local/lib/python3.11/site-packag

## Paso 3: dos modelos

In [5]:
evaluar("logistica", LogisticRegression(max_iter=1000), esquema)
evaluar("gradient boosting", GradientBoostingClassifier(random_state=42), esquema)

logistica            accuracy 0.9613   recall 0.2576   precision 0.8652


gradient boosting    accuracy 0.9712   recall 0.4610   precision 0.9051


## Paso 4: la matriz de confusión

In [6]:
pipeline = Pipeline([("preparar", preparacion()), ("modelo", GradientBoostingClassifier(random_state=42))])
probabilidades = cross_val_predict(pipeline, X, y, cv=esquema, method="predict_proba")[:, 1]
matriz = confusion_matrix(y, probabilidades >= 0.5, labels=[0, 1])
print("Matriz umbral 0.5 (filas reales; columnas predichas):", matriz, sep="\n")


Matriz umbral 0.5 (filas reales; columnas predichas):
[[5691   14]
 [ 159  136]]


## Paso 5: el umbral según el costo de cada error

In [7]:
COSTO_FRAUDE_NO_DETECTADO = 40
COSTO_FALSA_ALARMA = 3
UMBRALES = [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
costos = {}
detalle = []
for umbral in UMBRALES:
    tn, fp, fn, tp = confusion_matrix(y, probabilidades >= umbral, labels=[0, 1]).ravel()
    costos[umbral] = int(fn * COSTO_FRAUDE_NO_DETECTADO + fp * COSTO_FALSA_ALARMA)
    detalle.append(dict(umbral=umbral, tn=tn, fp=fp, fn=fn, tp=tp, costo=costos[umbral], costo_fp20=int(fn * 40 + fp * 20)))
umbral_elegido = min(costos, key=costos.get)
tabla_costos = pd.DataFrame(detalle)
print(pd.DataFrame(resultados.values()).to_string(index=False))
print(tabla_costos.to_string(index=False))
print("Umbral elegido:", umbral_elegido)
print("Con falsa alarma de $20:", tabla_costos.loc[tabla_costos.costo_fp20.idxmin(), "umbral"])
tabla_costos.to_csv("costos_fraude.csv", index=False)
pd.DataFrame(resultados.values()).to_csv("metricas_fraude.csv", index=False)


           modelo  accuracy  recall  precision
       linea base    0.9508  0.0000     0.0000
        logistica    0.9613  0.2576     0.8652
gradient boosting    0.9712  0.4610     0.9051
 umbral   tn  fp  fn  tp  costo  costo_fp20
   0.05 5117 588 100 195   5764       15760
   0.10 5483 222 122 173   5546        9320
   0.15 5584 121 135 160   5763        7820
   0.20 5630  75 145 150   6025        7300
   0.30 5666  39 150 145   6117        6780
   0.50 5691  14 159 136   6402        6640
Umbral elegido: 0.1
Con falsa alarma de $20: 0.5


## Comprobación

In [8]:
def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


def fila(nombre):
    return resultados.get(nombre)


comprobar(1, isinstance(esquema, StratifiedKFold) and esquema.shuffle and esquema.random_state == 42)
comprobar(2, fila("linea base") is not None and fila("linea base")["accuracy"] > 0.95 and fila("linea base")["recall"] == 0)
comprobar(3, fila("logistica") is not None and fila("gradient boosting") is not None and fila("gradient boosting")["recall"] > fila("logistica")["recall"])
comprobar(4, matriz is not None and matriz.sum() == len(pagos) and probabilidades is not None and len(probabilidades) == len(pagos))
comprobar(5, len(costos) == len(UMBRALES) and umbral_elegido == 0.1)

Paso 1: OK
Paso 2: OK
Paso 3: OK
Paso 4: OK
Paso 5: OK


## Si terminas antes

La línea base tiene más accuracy que una moneda al aire y no detecta ni un fraude. ¿Qué número de la tabla de `resultados` lo delata, y por qué la accuracy no puede?

El recall de la línea base es cero: pierde todos los fraudes aunque acierte la clase mayoritaria. Los umbrales se comparan sobre predicciones fuera de muestra, pero elegir y medir el mínimo con las mismas predicciones introduce optimismo. Para estimar el costo final se necesita un conjunto de prueba independiente o validación anidada.